# Title and Team Information

**Course:** ITCS 5123
**Project:** Interactive Netflix Catalog Dashboard
**Team:** Micah Platte and Smriti Bhemireddy
**Dataset:** Netflix Movies and Series, Kaggle
**Platform:** Python, Jupyter Notebook, Pandas, Plotly, ipywidgets, and scikit-learn
**Deliverables:** 2 case studies, 7 visualizations, 1 final notebook, and individual peer evaluations

Micah Platte is responsible for Case Study 1. Smriti Bhemireddy is responsible for Case Study 2. Data preparation, visualization 7, the dashboard, the literature connection, and the prompt log are shared.


# Project Introduction and Research Questions

This project explores how Netflix's content catalog varies across time, genres, countries, content types, and ratings. The work is one interactive notebook rather than a set of unrelated charts.

**Research questions**
1. How has Netflix's distribution of movies and TV shows changed over time?
2. Which genres are most common, and how has their representation changed?
3. Which countries contribute the most Netflix content?
4. How do genre and rating distributions differ across countries and content types?
5. Can movie characteristics reveal distinct content groups?

Seven required visualizations answer these questions. A rating-by-country heatmap completes question 4. Two optional diagrams, a country-genre connection drawing and a release-to-addition timeline, come after the required report sections.


# Dataset Description

The source file is the Netflix Movies and Series dataset published on Kaggle by debayank2024: https://www.kaggle.com/datasets/debayank2024/netflix-movies-and-series

The original file is `data/netflix_titles.csv`. The cleaned file used by every chart is `data/netflix_cleaned.csv`. The cleaned table has 8,806 titles: 6,130 movies and 2,676 TV shows. Titles were added from 2008 through 2021. Release years run from 1925 through 2021.

Each row is one title. The fields used in the charts are:

| Field | Meaning |
| --- | --- |
| `type` | Movie or TV Show |
| `title` | Title name |
| `country` | Producing countries, separated by commas |
| `date_added` | Date Netflix added the title |
| `release_year` | Year the title was released |
| `rating` | Content rating, such as TV-MA or PG-13 |
| `duration` | Minutes for movies, or seasons for TV shows |
| `listed_in` | Genres, separated by commas |
| `year_added` | Year parsed from `date_added` |
| `duration_minutes` | Runtime in minutes for movies |

Of the 8,806 titles, 7,975 list at least one country. The other 831 have country recorded as Unknown. A title can list more than one country and more than one genre, so country totals and genre totals are not exclusive shares of the catalog.


# Background Research and Literature Review

**Ali et al. (2025), Data-Driven Insights: Machine Learning Approaches for Netflix Content Analysis and Visualization.** The authors use Python and Tableau to compare Netflix content with pie charts, line graphs, histograms, and treemaps. They compare countries, movie and TV ratings, and genres, and they track how the catalog changes over time. The United States and India contribute the largest portions of content. TV-MA is one of the most common ratings. A line graph shows a large increase in titles added from 2014 to 2020. Case Study 1 uses that time comparison and separates movies from TV shows. The dashboard then adds filters for year, genre, content type, and rating so the same comparison can be narrowed.

**Hayadi and Priyanto (2025), Clustering Netflix Shows Based on Features Using K-means and Hierarchical Algorithms to Identify Content Patterns.** The authors cluster 5,185 Netflix movies. They fill missing values, one-hot encode rating, and standardize the numeric features. K-means with four clusters separates longer TV-14 movies, shorter TV-MA movies, a second TV-MA group with moderate runtimes, and longer R-rated movies. They also fit hierarchical clustering on a 1,000-movie sample and show a dendrogram. The paper has no viewership data, and the features are basic catalog fields. Case Study 2 repeats the K-means setup on this dataset and adds the country views the paper does not include. Hierarchical clustering is not repeated.

**Stolte, Tang, and Hanrahan (2002), Polaris: A System for Query, Analysis, and Visualization of Multidimensional Relational Databases.** https://graphics.stanford.edu/papers/polaris_extended/polaris.pdf Polaris treats a visualization as a specification that maps table fields onto visual encodings and can be driven by a query. The year, content type, genre, country, and rating controls work that way. Each control is one field in a single query, and every chart is redrawn from the rows that query returns.

**Liu, Jiang, and Heer (2013), imMens: Real-time Visual Querying of Big Data.** https://datavis.cs.columbia.edu/files/papers/immens.pdf imMens links several views so a selection updates all of them together. This dashboard does not brush inside a chart. It links the charts through one filter state. Changing any control refreshes every chart from the same subset. Cluster labels stay fixed, because they were fit on the full movie set before filtering.


# Methodology and Data Preprocessing

`src/clean-data.py` builds `data/netflix_cleaned.csv` from the original Kaggle file. It strips extra whitespace, moves durations that were stored in the rating column back into `duration`, fills missing director, cast, country, and rating values with Unknown, parses `date_added`, splits duration into minutes for movies and seasons for TV shows, and drops the duplicate id `s6530`.

The notebook then splits `country` and `listed_in` on commas into `country_list` and `genre_list`. Charts count a co-production once for every listed country and a title once for every listed genre.

Movie clusters for Visualization 6 are fit once, before any filter is applied. The features are duration in minutes, release year, and a one-hot encoding of rating. Those columns are standardized. K-means uses four clusters, `n_init=10`, and `random_state=42`, matching Hayadi and Priyanto. TV shows, and the five movies missing a duration, release year, or known rating, are labeled Not Classified. Dashboard filters hide points. They do not refit the clusters.

The notebook finds the project folder by looking for `data/netflix_cleaned.csv` in the current directory and then in its parent. Chart code does not use a machine-specific path.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets

from pathlib import Path
from IPython.display import display, clear_output, HTML
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Find the project root
project_root = Path.cwd()

if not (project_root / "data" / "netflix_cleaned.csv").exists():
    project_root = project_root.parent

# Load cleaned dataset
file_path = project_root / "data" / "netflix_cleaned.csv"

df = pd.read_csv(file_path)

# Ensure numerical columns are correctly typed
df["year_added"] = pd.to_numeric(
    df["year_added"], errors="coerce"
)

df["release_year"] = pd.to_numeric(
    df["release_year"], errors="coerce"
)

df["duration_minutes"] = pd.to_numeric(
    df["duration_minutes"], errors="coerce"
)

# Prepare categorical columns
for col in ["type", "rating", "country", "listed_in"]:
    df[col] = df[col].fillna("Unknown").astype(str).str.strip()

# Split multiple genres and countries
df["genre_list"] = df["listed_in"].apply(
    lambda s: [x.strip() for x in s.split(",") if x.strip()]
)

df["country_list"] = df["country"].apply(
    lambda s: [x.strip() for x in s.split(",") if x.strip()]
)

print("Dataset loaded successfully!")
print("Total titles:", len(df))
print("Movies:", (df["type"] == "Movie").sum())
print("TV Shows:", (df["type"] == "TV Show").sum())

df.head()


Dataset loaded successfully!
Total titles: 8806
Movies: 6130
TV Shows: 2676


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,duration_number,duration_minutes,seasons,genre_list,country_list
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9.0,90,90.0,NaN,[Documentaries],[United States]
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,9.0,2,NaN,2.0,"[International TV Shows, TV Dramas, TV Mysteries]",[South Africa]
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,9.0,1,NaN,1.0,"[Crime TV Shows, International TV Shows, TV Ac...",[Unknown]
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,9.0,1,NaN,1.0,"[Docuseries, Reality TV]",[Unknown]
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,9.0,2,NaN,2.0,"[International TV Shows, Romantic TV Shows, TV...",[India]


## Movie Clusters for Visualization 6
Visualization 6 colors movies by a K-means cluster fit once on the full movie set. TV shows, and movies missing duration, release year, or rating, are labeled "Not Classified". Rating is one-hot encoded, duration and release year stay numeric, and all features are standardized. K-means uses four clusters, matching Hayadi and Priyanto (2025). Filters later in this notebook hide points. They do not refit the clusters.


In [2]:
df["cluster"] = "Not Classified"

usable = (
    df["type"].eq("Movie")
    & df["duration_minutes"].notna()
    & df["release_year"].notna()
    & df["rating"].ne("Unknown")
)

movies = df.loc[usable].copy()
rating_dummies = pd.get_dummies(movies["rating"])
features = pd.concat(
    [
        movies[["duration_minutes", "release_year"]].reset_index(drop=True),
        rating_dummies.reset_index(drop=True),
    ],
    axis=1,
)
scaled = StandardScaler().fit_transform(features)
labels = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(scaled)

movies = movies.copy()
movies["cluster_id"] = labels
cluster_names = {}
for cluster_id, group in movies.groupby("cluster_id"):
    top_rating = group["rating"].value_counts().idxmax()
    mean_minutes = group["duration_minutes"].mean()
    mean_year = group["release_year"].mean()
    cluster_names[cluster_id] = (
        f"Mostly {top_rating} (~{mean_minutes:.0f} min, {mean_year:.0f})"
    )

df.loc[movies.index, "cluster"] = movies["cluster_id"].map(cluster_names)

print("Classified movies:", int(usable.sum()))
print(df.loc[usable, "cluster"].value_counts())


Classified movies: 6125
cluster
Mostly TV-MA (~92 min, 2015)     3361
Mostly TV-14 (~110 min, 2012)    1477
Mostly R (~107 min, 2010)         797
Mostly PG-13 (~108 min, 2009)     490
Name: count, dtype: int64


# Case Study 1 — Netflix Catalog Evolution

**Author:** Micah Platte
**Paper:** Ali et al. (2025)

This case study asks how Netflix's catalog changed as titles were added, which genres grew, and how ratings differ between movies and TV shows. The three charts below are the case study. Each one states the visual mapping, shows the code and the figure, and interprets the result. Years before 2016 are sparse, so Visualizations 1 and 2 are drawn for 2016 through 2021. The earlier rows remain in the dataset.


## Visualization 1: Movies vs. TV Shows Over Time
**Research question:** How has the number of movies and TV shows added to Netflix changed over time?

**Visual mapping:** This is a stacked area chart. The horizontal position is the year a title was added. The height of each colored band is the number of titles added that year. Color separates movies from TV shows. Stacking makes the total additions and the split between the two types visible in one view. Ali et al. (2025) used a line graph for the rise in additions from 2014 to 2020. This chart keeps that time comparison and adds content type as a second field.

**Scope:** Years before 2016 have few titles (24 in 2014 and 82 in 2015, against 429 in 2016). The chart below, and the default dashboard slider, start at 2016 so the trend is easier to read. Earlier rows stay in the dataset.

**Findings:** From 2016 through 2021, movies account for more additions than TV shows in every year. Both types peak in 2019 and then decline. That rise matches Ali et al., who found a large increase in additions from 2014 to 2020. The drop after 2019 is a drop in titles added per year, not a count of titles removed from Netflix.


In [3]:
def create_chart_1(data):
    counts = (
        data.dropna(subset=["year_added"])
        .groupby(["year_added", "type"])
        .size()
        .reset_index(name="count")
    )

    fig = px.area(
        counts,
        x="year_added",
        y="count",
        color="type",
        title="1. Movies vs. TV Shows Added Over Time",
        labels={
            "year_added": "Year Added",
            "count": "Number of Titles",
            "type": "Content Type"
        }
    )

    return fig

added = df.dropna(subset=["year_added"])
added = added[added["year_added"].between(2016, 2021)]
fig = create_chart_1(added)
fig.show()


## Visualization 2: Genre Trends Over Time
**Research question:** Which genres are most common, and how has their representation changed?

**Visual mapping:** This is a heatmap. Each row is one of the 10 most common genres in the data being shown. Each column is a year added. Color is the number of titles, and the cell text repeats that count. Darker cells are larger counts. A title can carry more than one genre, so a title can appear in more than one row. Ali et al. compared genres with static charts. This heatmap adds the year, so a genre's rank can be followed across time.

**Scope:** The displayed chart uses titles added from 2016 through 2021, for the same reason as Visualization 1.

**Findings:** International Movies, Dramas, and Comedies stay at the top of the heatmap. International Movies peak in 2018 at 668 titles. Dramas peak in 2019 at 563, and Comedies peak in 2019 at 420. Romantic Movies and Children & Family Movies keep growing into 2020. Most genres then fall by 2021, which matches the decline in Visualization 1, but the peaks are not all in the same year.


In [4]:
def create_chart_2(data):
    genres = data.explode("genre_list").copy()

    top = genres["genre_list"].value_counts().head(10).index
    genres = genres[genres["genre_list"].isin(top)]

    counts = (
        genres.groupby(["genre_list", "year_added"])
        .size()
        .unstack(fill_value=0)
    )

    counts.columns = counts.columns.astype(int)

    fig = px.imshow(
        counts,
        title="2. Genre Trends Over Time",
        labels={
            "x": "Year Added",
            "y": "Genre",
            "color": "Number of Titles"
        },
        aspect="auto",
        text_auto=True,
        color_continuous_scale="Blues"
    )

    fig.update_layout(height=550)

    return fig

added = df.dropna(subset=["year_added"])
added = added[added["year_added"].between(2016, 2021)]
fig = create_chart_2(added)
fig.show()


## Visualization 3: Content Ratings by Type
**Research question:** How do content-rating distributions differ between movies and TV shows?

**Visual mapping:** This is a 100% stacked bar. The horizontal position is content type. The vertical position is the percentage of that type, so both bars run from 0 to 100 even though there are more movies than TV shows. Color is the content rating. Comparing percentages, rather than raw counts, is what makes the two catalogs comparable. Ali et al. identified TV-MA as one of the most common ratings. This chart checks that claim separately for movies and for TV shows.

**Findings:** TV-MA is the largest rating in both groups: about 34% of movies and about 43% of TV shows. TV-14 is second in both. R-rated titles are a visible share of movies, and TV-PG is a visible share of TV shows. Mature and teen ratings dominate both types, and the TV-show bar is more concentrated in TV-MA.


In [5]:
def create_chart_3(data):
    counts = (
        data.groupby(["type", "rating"])
        .size()
        .reset_index(name="count")
    )

    counts["percentage"] = (
        counts["count"]
        / counts.groupby("type")["count"].transform("sum")
        * 100
    )

    fig = px.bar(
        counts,
        x="type",
        y="percentage",
        color="rating",
        title="3. Content Ratings by Type",
        labels={
            "type": "Content Type",
            "percentage": "Percentage of Titles",
            "rating": "Content Rating"
        }
    )

    fig.update_layout(
        barmode="stack",
        yaxis_range=[0, 100]
    )

    return fig

fig = create_chart_3(df)
fig.show()


# Case Study 2 — Geographic and Content Patterns

**Author:** Smriti Bhemireddy
**Paper:** Hayadi and Priyanto (2025)

This case study asks which countries supply the catalog, how genre and rating mix differs across those countries and across content types, and whether duration, release year, and rating separate movies into groups. The country charts extend Ali et al. The cluster chart repeats the K-means design from Hayadi and Priyanto. The full written comparison is in the visualization sections below and in `notebooks/case-study-two.ipynb`.


## Visualization 4: Country Contributions
**Research question:** Which countries contribute the most Netflix titles?

Bar length is the number of unique titles that list the country. Co-productions are counted in each listed country, and titles with country "Unknown" are excluded. On the full catalog, 3,690 titles (46.3% of titles with a country) list the United States, 1,046 (13.1%) list India, and 806 (10.1%) list the United Kingdom. The dashboard year slider starts at 2016, so the chart in the dashboard shows that shorter window until the slider is widened.

**Visual mapping:** Bar length is the number of unique titles that list the country. Color repeats that count. A horizontal bar keeps the country names readable.

**Research connection:** Ali et al. (2025) reported that the United States and India contributed the largest shares of Netflix content. This chart checks that claim on this dataset. Hayadi and Priyanto (2025) did not rank countries.


In [6]:
def create_chart_4(data):
    countries = data.explode("country_list").copy()
    countries["country_list"] = countries["country_list"].astype(str).str.strip()
    countries = countries[~countries["country_list"].isin(["Unknown", ""])]
    countries = countries.drop_duplicates(["show_id", "country_list"])

    if countries.empty:
        fig = go.Figure()
        fig.update_layout(
            title="4. Countries by Number of Netflix Titles",
            annotations=[dict(
                text="No country data available.",
                showarrow=False,
                xref="paper",
                yref="paper",
                x=0.5,
                y=0.5,
            )],
            xaxis_visible=False,
            yaxis_visible=False,
        )
        return fig

    counts = (
        countries.groupby("country_list")["show_id"]
        .nunique()
        .sort_values(ascending=False)
        .head(12)
        .sort_values()
        .reset_index()
    )
    counts.columns = ["country", "count"]

    fig = px.bar(
        counts,
        x="count",
        y="country",
        orientation="h",
        title="4. Countries by Number of Netflix Titles",
        labels={"count": "Number of Titles", "country": "Country"},
        text="count",
        color="count",
        color_continuous_scale="Blues",
    )
    fig.update_layout(
        height=550,
        showlegend=False,
        coloraxis_showscale=False,
        xaxis_title="Number of Titles",
        yaxis_title="Country",
    )
    fig.update_traces(textposition="outside", cliponaxis=False)
    return fig

fig = create_chart_4(df)
fig.show()


## Visualization 5: Country–Genre Relationships
**Research question:** How does genre mix differ across major producing countries?

Color is the percentage of that country's titles carrying the genre. Percentages are used instead of raw counts so a large producer does not dominate every cell only because it has more titles. Rows can sum to more than 100% because titles have multiple genres.

On the full catalog, International Movies account for 82.6% of Indian titles and 4.5% of U.S. titles. International TV Shows account for 65.8% of South Korean titles and 47.5% of Japanese titles, compared with 2.0% of U.S. titles.

**Visual mapping:** Rows are countries, columns are genres, and color is the percentage of that country's titles carrying the genre.

**Research connection:** Ali et al. (2025) compared countries and genres in separate charts. This heatmap shows the genre mix within each country. Hayadi and Priyanto (2025) did not study geography.


In [7]:
def create_chart_5(data):
    countries = data.explode("country_list").copy()
    countries["country_list"] = countries["country_list"].astype(str).str.strip()
    countries = countries[~countries["country_list"].isin(["Unknown", ""])]
    countries = countries.drop_duplicates(["show_id", "country_list"])

    if countries.empty:
        fig = go.Figure()
        fig.update_layout(
            title="5. Genre Mix by Country",
            annotations=[dict(
                text="No country data available.",
                showarrow=False,
                xref="paper",
                yref="paper",
                x=0.5,
                y=0.5,
            )],
            xaxis_visible=False,
            yaxis_visible=False,
        )
        return fig

    top_countries = (
        countries.groupby("country_list")["show_id"]
        .nunique()
        .sort_values(ascending=False)
        .head(8)
        .index
    )
    countries = countries[countries["country_list"].isin(top_countries)]
    country_totals = countries.groupby("country_list")["show_id"].nunique()

    genres = countries.explode("genre_list").copy()
    genres["genre_list"] = genres["genre_list"].astype(str).str.strip()
    genres = genres[~genres["genre_list"].isin(["", "Unknown"])]
    genres = genres.drop_duplicates(["show_id", "country_list", "genre_list"])

    top_genres = (
        genres.groupby("genre_list")["show_id"]
        .nunique()
        .sort_values(ascending=False)
        .head(8)
        .index
    )
    genres = genres[genres["genre_list"].isin(top_genres)]

    counts = (
        genres.groupby(["country_list", "genre_list"])["show_id"]
        .nunique()
        .unstack(fill_value=0)
    )
    counts = counts.reindex(index=top_countries, columns=top_genres, fill_value=0)
    percents = counts.div(country_totals.reindex(counts.index), axis=0) * 100

    fig = px.imshow(
        percents,
        title="5. Genre Mix by Country",
        labels={
            "x": "Genre",
            "y": "Country",
            "color": "Percent of Country's Titles",
        },
        aspect="auto",
        text_auto=".0f",
        color_continuous_scale="Blues",
    )
    fig.update_layout(height=560)
    return fig

fig = create_chart_5(df)
fig.show()


## Ratings and Genres across Countries and Content Types
**Research question:** How do genre and rating distributions differ across countries and content types?

Visualizations 3 and 5 answer part of this. Visualization 3 compares ratings for movies and TV shows. Visualization 5 compares genre mix across countries. Two pieces are still separate: ratings by country, and genres by content type.

**Rating mix by country.** The heatmap below uses the same percentage encoding as Visualization 5. Rows are the eight countries with the most titles. Columns are the most common ratings. Color is the share of that country's titles. A co-production is counted in each listed country.

On the full catalog: United States (3,690 titles): TV-MA 29.8%, R 17.9%, TV-14 13.5%; India (1,046 titles): TV-14 54.7%, TV-MA 25.4%, TV-PG 13.8%; United Kingdom (806 titles): TV-MA 31.4%, R 18.0%, TV-14 12.8%; Canada (445 titles): TV-MA 24.0%, R 17.8%, TV-14 11.0%; France (393 titles): TV-MA 41.5%, R 14.5%, TV-14 12.2%; Japan (318 titles): TV-MA 31.8%, TV-14 31.1%, TV-PG 15.7%; Spain (232 titles): TV-MA 73.3%, TV-14 7.8%, R 5.6%; South Korea (231 titles): TV-MA 39.8%, TV-14 37.2%, TV-PG 8.2%.

The United States, the United Kingdom, and Canada put a large share in TV-MA and another large share in R. India is different: TV-14 is the majority, at 54.7%. Spain is the most concentrated, with TV-MA on 73.3% of its titles. Japan and South Korea split more evenly between TV-MA and TV-14.

**Genre mix by content type.** Genre labels are mostly separate for movies and TV shows, so a shared genre heatmap would hide the split. Among 6,130 movies, International Movies cover 2,751 (44.9%) and Dramas cover 2,426 (39.6%). Among 2,676 TV shows, International TV Shows cover 1,351 (50.5%) and TV Dramas cover 763 (28.5%). The content-type filter on Visualization 7 shows this shift in the bar chart.


In [8]:
def _blank_figure(title, message):
    fig = go.Figure()
    fig.update_layout(
        title=title,
        annotations=[dict(
            text=message,
            showarrow=False,
            xref="paper",
            yref="paper",
            x=0.5,
            y=0.5,
        )],
        xaxis_visible=False,
        yaxis_visible=False,
    )
    return fig


def create_rating_by_country(data):
    countries = data.explode("country_list").copy()
    countries["country_list"] = countries["country_list"].astype(str).str.strip()
    countries = countries[~countries["country_list"].isin(["Unknown", ""])]
    countries = countries.drop_duplicates(["show_id", "country_list"])
    countries = countries[countries["rating"].ne("Unknown")]

    if countries.empty:
        return _blank_figure(
            "Rating Mix by Country",
            "No country and rating data available.",
        )

    top_countries = (
        countries.groupby("country_list")["show_id"]
        .nunique()
        .sort_values(ascending=False)
        .head(8)
        .index
    )
    countries = countries[countries["country_list"].isin(top_countries)]
    country_totals = countries.groupby("country_list")["show_id"].nunique()
    top_ratings = (
        countries.groupby("rating")["show_id"]
        .nunique()
        .sort_values(ascending=False)
        .head(6)
        .index
    )
    counts = (
        countries.groupby(["country_list", "rating"])["show_id"]
        .nunique()
        .unstack(fill_value=0)
    )
    counts = counts.reindex(index=top_countries, columns=top_ratings, fill_value=0)
    percents = counts.div(country_totals.reindex(counts.index), axis=0) * 100

    fig = px.imshow(
        percents,
        title="Rating Mix by Country",
        labels={
            "x": "Content Rating",
            "y": "Country",
            "color": "Percent of Country's Titles",
        },
        aspect="auto",
        text_auto=".0f",
        color_continuous_scale="Blues",
    )
    fig.update_layout(height=520)
    return fig


fig = create_rating_by_country(df)
fig.show()


## Visualization 6: Movie Attribute Clusters
**Research question:** Can duration, release year, and rating separate movies into distinct groups?

Each point is a movie. Position shows release year and duration. Color shows the cluster assigned above. The largest group is shorter and mostly TV-MA (3,361 movies, about 92 minutes). A longer TV-14 group (1,477 movies, about 110 minutes), an R-rated group (797 movies), and a PG-13 group (490 movies) make up the rest. These groups overlap on the scatterplot because rating separates them more than duration or year does.

**Visual mapping:** Horizontal position is release year, vertical position is duration in minutes, and color is the K-means cluster.

**Research connection:** Hayadi and Priyanto (2025) clustered 5,185 movies on duration, release year, and rating. Their four groups were longer TV-14 films, shorter TV-MA films, a second moderate-length TV-MA group, and longer R-rated films. This scatterplot uses those same features and k = 4.


In [9]:
def create_chart_6(data):
    movies = data[
        (data["type"] == "Movie")
        & (data["cluster"] != "Not Classified")
    ].copy()

    if movies.empty:
        fig = go.Figure()
        fig.update_layout(
            title="6. Movie Clusters by Duration and Release Year",
            annotations=[dict(
                text="No classified movies in this subset.",
                showarrow=False,
                xref="paper",
                yref="paper",
                x=0.5,
                y=0.5,
            )],
            xaxis_visible=False,
            yaxis_visible=False,
        )
        return fig

    order = movies["cluster"].value_counts().index.tolist()
    fig = px.scatter(
        movies,
        x="release_year",
        y="duration_minutes",
        color="cluster",
        category_orders={"cluster": order},
        title="6. Movie Clusters by Duration and Release Year",
        labels={
            "release_year": "Release Year",
            "duration_minutes": "Duration (minutes)",
            "cluster": "Cluster",
        },
        opacity=0.55,
        hover_data=["title", "rating"],
    )
    fig.update_layout(height=560)
    return fig

fig = create_chart_6(df)
fig.show()


# Integrated Dashboard and Visualization 7

Visualization 7 is the shared genre overview. The dashboard then places all seven required charts, the rating-by-country heatmap, and the two bonus diagrams under one set of filters. The filters are year range, content type, genre, country, and content rating. A reset button restores the 2016–2021 window and sets the other controls to All. Every chart is redrawn from the same filtered rows. This is the Polaris query pattern and a simpler form of the linked views in imMens: the link is the shared filter state, not a brush on a chart.


### Visualization 7: Overall Genre Distribution

This visualization examines the most frequently represented genres in Netflix's content catalog. A horizontal bar chart is used to compare the number of titles associated with each genre, with longer bars representing higher counts. The visualization displays the twelve most common genre labels and allows users to observe how their distribution changes when dashboard filters are applied.

Since individual titles may belong to multiple genres, each title can contribute to more than one category. As a result, the combined genre counts may exceed the total number of unique titles in the selected dataset.

On the full catalog, the three most common genres are International Movies (2,751 titles), Dramas (2,426), and Comedies (1,674). The bars shorten quickly after those labels, so a small set of genres accounts for most of the tags in the file.

**Visual mapping:** Bar length is the number of titles carrying the genre. The chart keeps the 12 most common labels. Color repeats the count.

**Research connection:** Ali et al. (2025) identified common genres with static charts. This shared chart is the overview that links Case Study 1 and Case Study 2, and the dashboard filters it with the other views.


In [10]:
def create_chart_7(data):
    genres = data.explode("genre_list").copy()

    genres = genres.drop_duplicates(
        ["show_id", "genre_list"]
    )

    counts = (
        genres["genre_list"]
        .value_counts()
        .head(12)
        .sort_values()
        .reset_index()
    )

    counts.columns = ["genre", "count"]

    fig = px.bar(
        counts,
        x="count",
        y="genre",
        orientation="h",
        title="7. Most Common Netflix Genres",
        labels={
            "count": "Number of Titles",
            "genre": "Genre"
        },
        text="count",
        color="count",
        color_continuous_scale="Blues"
    )

    fig.update_layout(
        height=550,
        showlegend=False,
        coloraxis_showscale=False,
        xaxis_title="Number of Titles",
        yaxis_title="Genre"
    )

    fig.update_traces(
        textposition="outside",
        cliponaxis=False
    )

    return fig

In [11]:
fig = create_chart_7(df)
fig.show()

## Final Interactive Dashboard

In [12]:
# Get available years
years = sorted(
    df["year_added"].dropna().astype(int).unique()
)

min_year = int(min(years))
max_year = int(max(years))

# Collect available categories
all_genres = sorted({
    genre
    for genres in df["genre_list"]
    for genre in genres
})

all_countries = sorted({
    country
    for countries in df["country_list"]
    for country in countries
    if country != "Unknown"
})

all_ratings = sorted(df["rating"].unique())

# Year range slider
year_filter = widgets.IntRangeSlider(
    value=(max(2016, min_year), max_year),
    min=min_year,
    max=max_year,
    step=1,
    description="Years:",
    continuous_update=False,
    layout=widgets.Layout(width="450px")
)

# Content type dropdown
type_filter = widgets.Dropdown(
    options=["All", "Movie", "TV Show"],
    value="All",
    description="Type:"
)

# Genre dropdown
genre_filter = widgets.Dropdown(
    options=["All"] + all_genres,
    value="All",
    description="Genre:"
)

# Country dropdown
country_filter = widgets.Dropdown(
    options=["All"] + all_countries,
    value="All",
    description="Country:"
)

# Rating dropdown
rating_filter = widgets.Dropdown(
    options=["All"] + all_ratings,
    value="All",
    description="Rating:"
)

In [13]:
def filter_data():
    filtered = df.copy()

    # Year range
    start_year, end_year = year_filter.value

    filtered = filtered[
        filtered["year_added"].between(
            start_year, end_year
        )
    ]
    if filtered.empty:
        return filtered

    # Content type
    if type_filter.value != "All":
        filtered = filtered[
            filtered["type"] == type_filter.value
        ]
        if filtered.empty:
            return filtered

    # Genre
    if genre_filter.value != "All":
        genre_mask = filtered["genre_list"].map(
            lambda genres: genre_filter.value in genres
        )
        filtered = filtered[genre_mask]
        if filtered.empty:
            return filtered

    # Country
    if country_filter.value != "All":
        country_mask = filtered["country_list"].map(
            lambda countries: country_filter.value in countries
        )
        filtered = filtered[country_mask]
        if filtered.empty:
            return filtered

    # Content rating
    if rating_filter.value != "All":
        filtered = filtered[
            filtered["rating"] == rating_filter.value
        ]

    return filtered


In [14]:
charts = [
    ("Figure 1", create_chart_1),
    ("Figure 2", create_chart_2),
    ("Figure 3", create_chart_3),
    ("Figure 4", create_chart_4),
    ("Figure 5", create_chart_5),
    ("Figure 6", create_chart_6),
    ("Figure 7", create_chart_7),
    ("Rating mix", create_rating_by_country),
]


In [15]:
dashboard_output = widgets.Output()

def update_dashboard(change=None):
    filtered = filter_data()

    with dashboard_output:
        clear_output(wait=True)

        # Dashboard title
        display(HTML(
            "<h2>Netflix Catalog Analysis Dashboard</h2>"
        ))

        # Summary statistics
        total = len(filtered)
        movies = (filtered["type"] == "Movie").sum()
        shows = (filtered["type"] == "TV Show").sum()

        display(HTML(
            f"<h3>Total Titles: {total:,} | "
            f"Movies: {movies:,} | "
            f"TV Shows: {shows:,}</h3>"
        ))

        if filtered.empty:
            display(HTML(
                "<p>No titles match the selected filters.</p>"
            ))
            return

        for name, create_fig in charts:
            # Check for chart-specific missing data
            if name in ["Figure 4", "Figure 5", "Rating mix", "Bonus 1"]:
                has_countries = filtered["country_list"].apply(
                    lambda countries: any(
                        c != "Unknown" for c in countries
                    )
                ).any()

                if not has_countries:
                    display(HTML(
                        f"<p>{name}: No country data available.</p>"
                    ))
                    continue

            if name == "Figure 6":
                has_clusters = (
                    (filtered["type"] == "Movie")
                    & (filtered["cluster"] != "Not Classified")
                ).any()

                if not has_clusters:
                    display(HTML(
                        "<p>Figure 6: No classified movies "
                        "match these filters.</p>"
                    ))
                    continue

            if name in ["Figure 2", "Figure 7"]:
                has_genres = filtered["genre_list"].apply(
                    lambda genres: len(genres) > 0
                ).any()

                if not has_genres:
                    display(HTML(
                        f"<p>{name}: No genre data available.</p>"
                    ))
                    continue

            fig = create_fig(filtered)
            display(fig)


In [16]:
# Register change handlers
for control in [
    year_filter,
    type_filter,
    genre_filter,
    country_filter,
    rating_filter
]:
    control.observe(update_dashboard, names="value")

# Reset button
reset_button = widgets.Button(
    description="Reset Filters",
    button_style="info"
)

def reset_filters(button):
    year_filter.value = (
        max(2016, min_year),
        max_year
    )
    type_filter.value = "All"
    genre_filter.value = "All"
    country_filter.value = "All"
    rating_filter.value = "All"

    update_dashboard()

reset_button.on_click(reset_filters)

# Organize filters
filter_layout = widgets.VBox([
    year_filter,
    widgets.HBox([type_filter, genre_filter]),
    widgets.HBox([country_filter, rating_filter]),
    reset_button
])

# Display dashboard
display(filter_layout, dashboard_output)

# Render initial charts
update_dashboard()

Output()

# LLM Integration and Prompt Documentation


The outline asks for the prompt, its purpose, a summary of the response, what was adopted, what was changed by hand, and the outcome. Full chatbot transcripts were not stored in the repository. The rows below use the prompts that are recorded in the project files and in this notebook's development. The peer evaluations are in the last section of this notebook.

| Exact prompt | Purpose | Summary of the response | Adopted | Changed by hand | Outcome |
| --- | --- | --- | --- | --- | --- |
| Analyze the dataset and identify necessary cleaning | Prepare one shared data file | Recorded in `src/clean-data.py` and in Case Study 1: strip text, move durations out of the rating field, fill missing categories, parse dates, split duration, drop duplicate `s6530` | Those steps are the cleaning script | The cleaned file was already saved and was not rebuilt | `data/netflix_cleaned.csv` |
| Recommend visualizations to display Netflix catalog trends | Choose charts for Case Study 1 | Case Study 1 records this as the chart-selection prompt | The area chart, genre heatmap, and stacked rating bar specified in the project outline | Chart types were kept as the outline specified them | Visualizations 1-3 |
| Debug and clean the base code to improve visualization quality | Fix Case Study 1 chart code | Case Study 1 records this as the prompt used for each visualization's code | The Case Study 1 chart code | The saved Case Study 1 notebook is the corrected version. The original broken code was not kept | Visualizations 1-3 run in `notebooks/case-study-one.ipynb` |
| ok do my parts and the shared stuff | Build Case Study 2 and the shared dashboard sections that were still empty | Charts for countries, country-genre mix, and K-means, plus Polaris, imMens, limitations, and this log | `create_chart_4`, `create_chart_5`, `create_chart_6`, the cluster column, and the sections under the dashboard | k stayed at 4 so the groups can be compared with Hayadi and Priyanto. Counts were computed from the cleaned file rather than taken from the model reply | `notebooks/case-study-two.ipynb` and the shared sections of this notebook |
| ok do everything but the peer evaluations then | Close the remaining outline gaps except peer evaluations | Write-ups beside visualizations 1-3, a rating-by-country heatmap, two custom bonus diagrams, this prompt table, and saved chart outputs | The cells added below Visualization 3 and below Visualization 7 | Bonus 2 omits year pairs with fewer than 8 titles so the drawing stays readable. The median wait still uses every dated title. Peer evaluations were left out, as requested | This notebook documents the seven required charts, the rating question, and both bonus diagrams |

The source for every chart is in this notebook.


# Results and Discussion



This project explored Netflix's content catalog through seven interactive visualizations, examining trends in content additions, genre representation, content ratings, geographic distribution, and movie characteristics. By combining two case studies into a single dashboard, we aimed to provide a more comprehensive understanding of how Netflix's content is distributed and how its composition has changed over time.

The first case study revealed that movies generally accounted for more annual additions than TV shows, with both categories experiencing significant growth leading up to 2019. International Movies, Dramas, and Comedies were among the most represented genres, following similar trends in annual additions. Content-rating comparisons also demonstrated that TV-MA represented a substantial portion of both movies and television shows. These observations generally aligned with the findings of Ali et al. (2025).

The second case study found a concentrated geography and a partial match to Hayadi and Priyanto (2025). Of titles with a country, 46.3% list the United States, 13.1% list India, and 10.1% list the United Kingdom. Genre mix differs by country: International Movies are 82.6% of Indian titles and 4.5% of U.S. titles, while International TV Shows dominate the South Korean and Japanese lists. K-means on duration, release year, and rating recovered a shorter TV-MA group, a longer TV-14 group, and a separate R-rated group. The paper's second TV-MA group did not appear. PG-13 formed its own cluster instead.

The shared genre chart shows the same concentration in a few labels. Across the full catalog, the three most common genres are International Movies (2,751 titles), Dramas (2,426), and Comedies (1,674). Because titles carry more than one genre, these counts are not exclusive.

By combining these findings into an interactive dashboard, users can filter and compare multiple aspects of the dataset to identify relationships that may not be immediately obvious in individual charts. Although the dataset represents a historical snapshot and contains some incomplete information, the project demonstrates how data visualization can help reveal meaningful patterns in Netflix's content distribution.

Ratings differ by country as well as by content type. United States (3,690 titles): TV-MA 29.8%, R 17.9%, TV-14 13.5%. India (1,046 titles): TV-14 54.7%, TV-MA 25.4%, TV-PG 13.8%. Spain (232 titles): TV-MA 73.3%, TV-14 7.8%, R 5.6%. Genre labels also split by content type: International Movies are 44.9% of movies, while International TV Shows are 50.5% of TV shows.

The two bonus diagrams use custom line encodings. The country-genre diagram maps line width to the number of titles in each pair. The release-to-addition diagram maps line width to how many titles share a release year and a year added, and it maps color to the wait. Among 8,796 titles with both years, the median wait is 1 year, 36.8% are added in the release year, and 13.7% wait more than 10 years.


# Limitations and Future Work


- The dataset records titles added through 2021. It does not show the current Netflix catalog or any viewership.
- 831 titles have no country, and some titles are missing a date added.
- Countries and genres are multi-valued, so one title can contribute to several counts. Country bars are not exclusive shares of the catalog.
- Movie clusters use only duration, release year, and rating. TV shows are excluded because duration is stored in seasons.
- Four clusters were used to match Hayadi and Priyanto, not because a quality score selected that k. Their hierarchical clustering was not repeated.
- The dashboard links charts through widgets. It does not support brushing directly on a chart, which is the interaction imMens demonstrates.

A later version could add the paper's hierarchical clustering, encode genre inside the movie clusters, and let a selection on one chart filter the others.
- The rating-by-country heatmap counts a co-production in every listed country.
- Bonus 1 draws only the eight largest countries and eight genres, so smaller producers are not in that diagram.
- Bonus 2 draws year pairs with at least 8 titles and release years from 1980 on. The median wait uses every title that has both years.


# References and Reproducibility Instructions

## References
Netflix Movies and Series dataset, Kaggle.
https://www.kaggle.com/datasets/debayank2024/netflix-movies-and-series

Ali et al. (2025). Data-Driven Insights: Machine Learning Approaches for Netflix Content Analysis and Visualization.

Hayadi, B. H., & Priyanto, E. (2025). Clustering Netflix shows based on features using K-means and hierarchical algorithms to identify content patterns. *International Journal for Applied Information Management, 5*(2), 98–110. https://doi.org/10.47738/ijaim.v5i2.102

Stolte, C., Tang, D., & Hanrahan, P. (2002). Polaris: A system for query, analysis, and visualization of multidimensional relational databases. *IEEE Transactions on Visualization and Computer Graphics, 8*(1), 52–65. https://graphics.stanford.edu/papers/polaris_extended/polaris.pdf

Liu, Z., Jiang, B., & Heer, J. (2013). imMens: Real-time visual querying of big data. *Computer Graphics Forum, 32*(3), 421–430. https://datavis.cs.columbia.edu/files/papers/immens.pdf

## Reproducibility instructions
1. Use the project virtual environment. From the project folder: `py -m venv .venv`, then `.\.venv\Scripts\python.exe -m pip install -r requirements.txt`.
2. Open this notebook and select that environment as the kernel.
3. Run all cells from top to bottom. The data path resolves whether the notebook is started in the project folder or in `notebooks`.
4. The original file is `data/netflix_titles.csv`. The cleaned file is `data/netflix_cleaned.csv`. Cleaning is `src/clean-data.py`.
5. Required libraries are listed in `requirements.txt`: pandas, plotly, ipywidgets, scikit-learn, and the Jupyter kernel packages.
6. The code cell after the optional bonus diagrams redraws every chart, writes an interactive HTML file for each one to `html/`, and checks that an empty filter still returns a table with the original columns. That cell follows the bonus functions because it draws those diagrams too.

The HTML files use Plotly's CDN script, so open them in a browser with a network connection. The figures saved in this notebook do not need that connection.


# Optional Bonus Visualizations

These two diagrams are the optional charts in the project outline. They are not part of the seven required visualizations. Both are drawn from positions, line widths, and colors set in the code. They do not call a built-in link or timeline chart.

Run the dashboard after this section if you want these two diagrams to respond to the same filters. The copies below use the full catalog.


## Bonus 1: Country-Genre Connection Diagram
**Research question:** Which country and genre pairs account for the most titles?

**Visual mapping:** This diagram is drawn from positions and line widths, not from a built-in link chart. Countries sit on the left and genres sit on the right. A line connects a country to a genre only when that pair occurs. Line width is the number of unique titles in the pair, scaled to the thickest pair in the view. Color identifies the country so a country's links can be followed across the diagram. The view keeps the eight countries with the most titles and the eight genres most common among those titles.

**Findings:** The thickest pairs are India to International Movies (864 titles); United States to Dramas (835 titles); United States to Comedies (680 titles); India to Dramas (662 titles); United States to Documentaries (512 titles). The United States spreads across several thick links. India's thickest links are International Movies and Dramas, which matches Visualization 5.


In [17]:
def create_bonus_connections(data):
    countries = data.explode("country_list").copy()
    countries["country_list"] = countries["country_list"].astype(str).str.strip()
    countries = countries[~countries["country_list"].isin(["Unknown", ""])]
    countries = countries.drop_duplicates(["show_id", "country_list"])

    if countries.empty:
        return _blank_figure(
            "Bonus 1. Country-Genre Connections",
            "No country data available.",
        )

    top_countries = (
        countries.groupby("country_list")["show_id"]
        .nunique()
        .sort_values(ascending=False)
        .head(8)
        .index
        .tolist()
    )
    countries = countries[countries["country_list"].isin(top_countries)]
    genres = countries.explode("genre_list").copy()
    genres["genre_list"] = genres["genre_list"].astype(str).str.strip()
    genres = genres[~genres["genre_list"].isin(["", "Unknown"])]
    genres = genres.drop_duplicates(["show_id", "country_list", "genre_list"])

    if genres.empty:
        return _blank_figure(
            "Bonus 1. Country-Genre Connections",
            "No genre data available.",
        )

    top_genres = (
        genres.groupby("genre_list")["show_id"]
        .nunique()
        .sort_values(ascending=False)
        .head(8)
        .index
        .tolist()
    )
    genres = genres[genres["genre_list"].isin(top_genres)]
    pair_counts = (
        genres.groupby(["country_list", "genre_list"])["show_id"]
        .nunique()
    )
    max_count = int(pair_counts.max())
    palette = px.colors.qualitative.Dark2
    country_y = {
        country: 1 - i / max(len(top_countries) - 1, 1)
        for i, country in enumerate(top_countries)
    }
    genre_y = {
        genre: 1 - i / max(len(top_genres) - 1, 1)
        for i, genre in enumerate(top_genres)
    }

    fig = go.Figure()
    for country_index, country in enumerate(top_countries):
        color = palette[country_index % len(palette)]
        for genre in top_genres:
            value = int(pair_counts.get((country, genre), 0))
            if value == 0:
                continue
            fig.add_trace(go.Scatter(
                x=[0, 1],
                y=[country_y[country], genre_y[genre]],
                mode="lines",
                line=dict(width=1 + 16 * value / max_count, color=color),
                hovertemplate=(
                    f"{country} to {genre}: {value} titles<extra></extra>"
                ),
                showlegend=False,
            ))

    fig.add_trace(go.Scatter(
        x=[0] * len(top_countries),
        y=[country_y[country] for country in top_countries],
        mode="markers+text",
        text=top_countries,
        textposition="middle left",
        marker=dict(
            size=11,
            color=[palette[i % len(palette)] for i in range(len(top_countries))],
        ),
        hoverinfo="skip",
        showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=[1] * len(top_genres),
        y=[genre_y[genre] for genre in top_genres],
        mode="markers+text",
        text=top_genres,
        textposition="middle right",
        marker=dict(size=11, color="#4d4d4d"),
        hoverinfo="skip",
        showlegend=False,
    ))
    fig.update_layout(
        title="Bonus 1. Country-Genre Connections",
        height=640,
        plot_bgcolor="white",
        xaxis=dict(visible=False, range=[-0.55, 1.62]),
        yaxis=dict(visible=False, range=[-0.08, 1.08]),
        margin=dict(l=140, r=180, t=60, b=30),
    )
    return fig


fig = create_bonus_connections(df)
fig.show()


## Bonus 2: Release-to-Addition Timeline
**Research question:** How long after a title's release year does Netflix add it?

**Visual mapping:** Each title has a release year and a year added. Those two years are placed on two vertical axes, release year on the left and year added on the right. A line joins the two years for titles that share the same pair. Line width is the number of titles in that pair. Color is the gap in years: blue is a short wait and orange is a wait of about 30 years. The axes, widths, and colors are set in the code rather than by a standard chart template. Pairs with fewer than 8 titles are left off the drawing so the thicker flows stay readable. The summary numbers below use every title that has both years.

**Findings:** Among 8,796 titles with both years, the median wait is 1 year. 36.8% are added in the same year they were released, and 13.7% wait more than 10 years. The thick lines sit near the diagonal of recent release years and recent add years. Thinner orange lines are older films added much later.


In [18]:
def create_bonus_timeline(data):
    rows = data.dropna(subset=["release_year", "year_added"]).copy()
    rows = rows[
        (rows["release_year"] >= 1980)
        & (rows["year_added"] >= 2008)
    ]
    if rows.empty:
        return _blank_figure(
            "Bonus 2. Release Year to Year Added",
            "No release year and year added are available.",
        )

    rows["release_year"] = rows["release_year"].astype(int)
    rows["year_added"] = rows["year_added"].astype(int)
    counts = (
        rows.groupby(["release_year", "year_added"])
        .size()
        .reset_index(name="count")
    )
    counts = counts[counts["count"] >= 8]
    if counts.empty:
        return _blank_figure(
            "Bonus 2. Release Year to Year Added",
            "No year pair has enough titles to draw.",
        )

    min_year = int(min(counts["release_year"].min(), counts["year_added"].min()))
    max_year = int(max(counts["release_year"].max(), counts["year_added"].max()))
    span = max(max_year - min_year, 1)
    max_count = int(counts["count"].max())

    def year_y(year):
        return (int(year) - min_year) / span

    def lag_color(lag):
        t = min(max(lag, 0), 30) / 30
        red = int(33 + 205 * t)
        green = int(102 + 50 * (1 - t))
        blue = int(172 * (1 - t) + 37 * t)
        return f"rgba({red},{green},{blue},0.55)"

    fig = go.Figure()
    for row in counts.itertuples(index=False):
        lag = int(row.year_added - row.release_year)
        fig.add_trace(go.Scatter(
            x=[0, 1],
            y=[year_y(row.release_year), year_y(row.year_added)],
            mode="lines",
            line=dict(width=0.6 + 14 * row.count / max_count, color=lag_color(lag)),
            hovertemplate=(
                f"Released {int(row.release_year)}, "
                f"added {int(row.year_added)}: "
                f"{int(row.count)} titles<extra></extra>"
            ),
            showlegend=False,
        ))

    tick_years = list(range(min_year - (min_year % 5), max_year + 1, 5))
    fig.add_trace(go.Scatter(
        x=[0] * len(tick_years) + [1] * len(tick_years),
        y=[year_y(year) for year in tick_years] * 2,
        mode="markers+text",
        text=[str(year) for year in tick_years] + [str(year) for year in tick_years],
        textposition=["middle left"] * len(tick_years) + ["middle right"] * len(tick_years),
        marker=dict(size=7, color="#333333"),
        hoverinfo="skip",
        showlegend=False,
    ))
    fig.update_layout(
        title="Bonus 2. Release Year to Year Added",
        height=720,
        plot_bgcolor="white",
        xaxis=dict(
            visible=False,
            range=[-0.28, 1.28],
        ),
        yaxis=dict(visible=False, range=[-0.04, 1.06]),
        annotations=[
            dict(text="Release year", x=0, y=1.04, xref="x", yref="paper", showarrow=False),
            dict(text="Year added", x=1, y=1.04, xref="x", yref="paper", showarrow=False),
        ],
        margin=dict(l=80, r=80, t=70, b=30),
    )
    return fig


fig = create_bonus_timeline(df)
fig.show()


In [19]:
charts.extend([
    ("Bonus 1", create_bonus_connections),
    ("Bonus 2", create_bonus_timeline),
])
update_dashboard()


The two functions above are now registered on the dashboard. The next cell redraws every chart, writes the HTML files, and checks the filters.


In [20]:
html_dir = project_root / "html"
html_dir.mkdir(exist_ok=True)

added = df.dropna(subset=["year_added"])
added = added[added["year_added"].between(2016, 2021)]

export_figures = {
    "01_movies_vs_tv_over_time": create_chart_1(added),
    "02_genre_trends_over_time": create_chart_2(added),
    "03_ratings_by_type": create_chart_3(df),
    "04_country_contributions": create_chart_4(df),
    "05_genre_mix_by_country": create_chart_5(df),
    "06_movie_clusters": create_chart_6(df),
    "07_overall_genres": create_chart_7(df),
    "08_rating_mix_by_country": create_rating_by_country(df),
    "bonus_1_country_genre_connections": create_bonus_connections(df),
    "bonus_2_release_to_addition": create_bonus_timeline(df),
}

for file_name, figure in export_figures.items():
    figure.write_html(
        html_dir / f"{file_name}.html",
        include_plotlyjs="cdn",
    )
    assert figure.data or figure.layout.annotations, file_name

full = filter_data()
assert "type" in full.columns
assert len(full) > 0

year_filter.value = (2008, 2008)
type_filter.value = "Movie"
genre_filter.value = "TV Comedies"
country_filter.value = "All"
rating_filter.value = "All"
empty = filter_data()
assert empty.empty
assert "type" in empty.columns

year_filter.value = (2016, 2021)
type_filter.value = "All"
genre_filter.value = "All"
update_dashboard()

print("Reproducibility check passed.")
print("Titles:", len(df))
print("HTML files:", len(list(html_dir.glob("*.html"))))


Reproducibility check passed.
Titles: 8806
HTML files: 10


# Peer Evaluations

| Team member | Rating | Why this rating |
| --- | ---: | --- |
| Micah Platte | 100% | She completed Case Study 1: visualizations 1–3, the Ali et al. comparison, the findings, and the limitations. She also completed her shared pieces: `src/clean-data.py`, the cleaned dataset, the dashboard filters and reset, and the visualization 7 chart function. |
| Smriti Bhemireddy | 100% | She completed Case Study 2: visualizations 4–6, the Hayadi and Priyanto comparison, and the rating-by-country analysis. She also completed the shared literature review, the prompt log, both bonus diagrams, and the final section order of this notebook. |
